<a href="https://colab.research.google.com/github/raymondtyaba/case-manager-dashboard/blob/main/Full_RAG_Studio_Capstone_Amended_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Full RAG Studio — Amended Pfizer Capstone
**John Raymond TYABA**

multiple uploaded supplier documents, OCR for scanned pages, manual 10-question sessions, one honest unsupported question, abstention, safe retrieval/generation fallbacks, citations, question counting, and session export.

In [1]:
!apt-get -qq update && apt-get -qq install -y tesseract-ocr poppler-utils
!pip -q install gradio==5.49.1 pypdf==6.1.1 sentence-transformers==5.1.1 faiss-cpu==1.12.0 transformers==4.57.1 sentencepiece==0.2.1 pdf2image==1.17.0 pytesseract==0.3.13 scikit-learn==1.7.2 pillow
print('Dependencies installed. If Colab asks for a runtime restart, restart and run all cells again.')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../poppler-utils_24.02.0-1ubuntu9.11_amd64.deb ...
Unpacking poppler-utils (24.02.0-1ubuntu9.11) ...
Setting up poppler-utils (24.02.0-1ubuntu9.11) ...
Processing triggers for man-db (2.12.0-4build2) ...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires pydantic<3,>=2.12, but you have pydantic 2.11.10 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.52.1 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 

In [ ]:
"""Full RAG Studio — amended Pfizer capstone.
Upload multiple digital/scanned PDFs, process them, then ask questions manually.
Preferred path: MiniLM + FAISS + FLAN-T5. Safe fallbacks: TF-IDF + extractive answer.
"""
from __future__ import annotations
import json, re, tempfile, time, traceback
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any
import numpy as np
import gradio as gr
from pypdf import PdfReader

@dataclass
class Chunk:
    text: str
    source: str
    source_id: str
    page_start: int
    page_end: int
    doc_type: str
    chunk_id: str

class DocumentProcessor:
    def __init__(self, chunk_size=900, overlap=150):
        self.chunk_size = chunk_size
        self.overlap = overlap

    @staticmethod
    def classify(text: str) -> str:
        low = text.lower()
        rules = {
            "Certificate of Quality": ["certificate of quality", "certificate of analysis", "assay", "batch", "lot number"],
            "Packaging Specification": ["packaging specification", "label", "carton", "blister"],
            "Compliance Declaration": ["compliance", "declaration", "regulation", "conformity"],
            "Research Report": ["abstract", "method", "results", "conclusion", "research"],
        }
        scores = {name: sum(term in low for term in terms) for name, terms in rules.items()}
        best = max(scores, key=scores.get)
        return best if scores[best] else "General Document"

    @staticmethod
    def normalise_files(files: Any) -> list[str]:
        if not files:
            return []
        items = files if isinstance(files, list) else [files]
        paths = []
        for item in items:
            if isinstance(item, str):
                paths.append(item)
            elif hasattr(item, "name"):
                paths.append(item.name)
            elif isinstance(item, dict) and item.get("path"):
                paths.append(item["path"])
        return [p for p in paths if p]

    @staticmethod
    def ocr_page(path: str, page_number: int) -> str:
        from pdf2image import convert_from_path
        import pytesseract
        images = convert_from_path(path, first_page=page_number, last_page=page_number, dpi=250)
        return pytesseract.image_to_string(images[0]) if images else ""

    def extract(self, path: str):
        reader = PdfReader(path)
        pages, ocr_pages = [], []
        for number, page in enumerate(reader.pages, 1):
            text = (page.extract_text() or "").strip()
            # OCR only when the page appears scanned or nearly empty.
            if len(re.sub(r"\s+", "", text)) < 40:
                try:
                    ocr_text = self.ocr_page(path, number).strip()
                    if ocr_text:
                        text = ocr_text
                        ocr_pages.append(number)
                except Exception:
                    pass
            pages.append((number, re.sub(r"\s+", " ", text).strip()))
        return pages, ocr_pages

    def process(self, paths: list[str]):
        chunks, summaries = [], []
        for file_no, path in enumerate(paths, 1):
            pages, ocr_pages = self.extract(path)
            source, source_id = Path(path).name, f"DOC-{file_no:03d}"
            for page_no, text in pages:
                if not text:
                    continue
                doc_type = self.classify(text)
                start, chunk_no = 0, 0
                while start < len(text):
                    end = min(start + self.chunk_size, len(text))
                    chunk_no += 1
                    chunks.append(Chunk(
                        text=text[start:end], source=source, source_id=source_id,
                        page_start=page_no, page_end=page_no, doc_type=doc_type,
                        chunk_id=f"{source_id}-P{page_no}-C{chunk_no}"
                    ))
                    if end == len(text):
                        break
                    start = max(end - self.overlap, start + 1)
            summaries.append({"file": source, "pages": len(pages), "ocr_pages": ocr_pages})
        return chunks, summaries

class RAGEngine:
    def __init__(self):
        self.chunks = []
        self.embedder = None
        self.faiss_index = None
        self.tfidf = None
        self.tfidf_matrix = None
        self.generator = None
        self.retrieval_backend = "not indexed"
        self.generation_backend = "not loaded"

    def build_index(self, chunks: list[Chunk]):
        if not chunks:
            raise ValueError("No readable text was extracted from the uploaded PDFs.")
        self.chunks = list(chunks)
        texts = [c.text for c in chunks]
        # Preferred semantic route. If model download or FAISS fails, index using TF-IDF.
        try:
            import faiss
            from sentence_transformers import SentenceTransformer
            self.embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
            vectors = self.embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False)
            vectors = np.asarray(vectors, dtype="float32")
            self.faiss_index = faiss.IndexFlatIP(vectors.shape[1])
            self.faiss_index.add(vectors)
            self.tfidf = self.tfidf_matrix = None
            self.retrieval_backend = "MiniLM embeddings + FAISS"
        except Exception as exc:
            from sklearn.feature_extraction.text import TfidfVectorizer
            self.tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
            self.tfidf_matrix = self.tfidf.fit_transform(texts)
            self.embedder = self.faiss_index = None
            self.retrieval_backend = f"TF-IDF fallback ({type(exc).__name__})"

    def retrieve(self, query: str, k=4, document_type="All", auto_route=True):
        if not self.chunks:
            return []
        allowed = list(range(len(self.chunks)))
        routed = DocumentProcessor.classify(query) if auto_route else "General Document"
        requested = document_type if document_type != "All" else (routed if routed != "General Document" else None)
        if requested:
            filtered = [i for i in allowed if self.chunks[i].doc_type == requested]
            if filtered:
                allowed = filtered
        if self.faiss_index is not None and self.embedder is not None:
            vector = np.asarray(self.embedder.encode([query], normalize_embeddings=True), dtype="float32")
            scores, ids = self.faiss_index.search(vector, len(self.chunks))
            ranked = [(int(i), float(s)) for s, i in zip(scores[0], ids[0]) if int(i) in allowed]
        else:
            from sklearn.metrics.pairwise import cosine_similarity
            scores = cosine_similarity(self.tfidf.transform([query]), self.tfidf_matrix).flatten()
            ranked = sorted(((i, float(scores[i])) for i in allowed), key=lambda x: x[1], reverse=True)
        return [(self.chunks[i], score) for i, score in ranked[:int(k)]]

    @staticmethod
    def extractive_answer(query: str, hits):
        terms = {w for w in re.findall(r"[a-zA-Z0-9%°<>.-]+", query.lower()) if len(w) > 2}
        candidates = []
        for chunk, similarity in hits:
            for sentence in re.split(r"(?<=[.!?])\s+", chunk.text):
                lexical = sum(term in sentence.lower() for term in terms)
                if lexical:
                    candidates.append((lexical, similarity, sentence.strip()))
        candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
        return " ".join(item[2] for item in candidates[:3]) if candidates else hits[0][0].text[:600]

    def generate(self, query: str, hits):
        context = "\n\n".join(
            f"[{n}] {chunk.source}, page {chunk.page_start}, {chunk.doc_type}:\n{chunk.text}"
            for n, (chunk, _) in enumerate(hits, 1)
        )
        prompt = (
            "Answer only from the context. If evidence is insufficient, say so. "
            "Be concise and cite [1], [2].\n\n"
            f"CONTEXT:\n{context}\n\nQUESTION: {query}\nANSWER:"
        )
        try:
            if self.generator is None:
                from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
                name = "google/flan-t5-small"
                tokenizer = AutoTokenizer.from_pretrained(name)
                model = AutoModelForSeq2SeqLM.from_pretrained(name)
                self.generator = pipeline("text2text-generation", model=model, tokenizer=tokenizer, device=-1)
            self.generation_backend = "FLAN-T5-small"
            output = self.generator(prompt, max_new_tokens=180, do_sample=False, truncation=True)[0]["generated_text"].strip()
            return output or self.extractive_answer(query, hits)
        except Exception as exc:
            self.generation_backend = f"extractive fallback ({type(exc).__name__})"
            return self.extractive_answer(query, hits)

    def answer(self, query: str, k=4, document_type="All", auto_route=True, threshold=0.05):
        hits = self.retrieve(query, k, document_type, auto_route)
        if not hits or hits[0][1] < threshold:
            self.generation_backend = "abstention"
            return {
                "answer": "I could not find enough evidence in the uploaded documents to answer this question.",
                "sources": [], "confidence": 0.0, "chunks_used": 0, "abstained": True
            }
        answer = self.generate(query, hits)
        sources = [{
            "source": c.source, "page": c.page_start, "document_type": c.doc_type,
            "chunk_id": c.chunk_id, "score": round(float(score), 3)
        } for c, score in hits]
        confidence = max(0.0, min(1.0, float(np.mean([max(0.0, s) for _, s in hits]))))
        return {"answer": answer, "sources": sources, "confidence": confidence,
                "chunks_used": len(hits), "abstained": False}

PROCESSOR = DocumentProcessor()
ENGINE = RAGEngine()
SESSION_LOG = []
DOCUMENT_TYPES = ["All", "Certificate of Quality", "Packaging Specification", "Compliance Declaration", "Research Report", "General Document"]

def process_ui(files):
    try:
        paths = PROCESSOR.normalise_files(files)
        if not paths:
            return "⚠️ Upload at least one PDF.", [], "", "Questions asked: **0**"
        started = time.perf_counter()
        chunks, summaries = PROCESSOR.process(paths)
        ENGINE.build_index(chunks)
        SESSION_LOG.clear()
        detail = "\n".join(f"- {s['file']}: {s['pages']} pages; OCR pages {s['ocr_pages'] or 'none'}" for s in summaries)
        status = (f"✅ Indexed **{len(paths)} file(s)**, **{sum(s['pages'] for s in summaries)} pages**, "
                  f"**{len(chunks)} chunks** in **{time.perf_counter()-started:.1f}s**.\n\n{detail}\n\n"
                  f"Retrieval backend: **{ENGINE.retrieval_backend}**")
        return status, [], "", "Questions asked: **0**"
    except Exception as exc:
        traceback.print_exc()
        return f"❌ {type(exc).__name__}: {exc}", [], "", "Questions asked: **0**"

def chat_ui(message, history, k, document_type, auto_route, threshold):
    # Chatbot type='messages': every history item is a dictionary with role/content.
    history = list(history or [])
    if not message or not message.strip():
        return history, "", f"Questions asked: **{len(SESSION_LOG)}**"
    try:
        result = ENGINE.answer(message.strip(), int(k), document_type, bool(auto_route), float(threshold))
        source_text = "\n".join(
            f"- {s['source']}, page {s['page']} | {s['document_type']} | {s['chunk_id']} | score {s['score']:.3f}"
            for s in result["sources"]
        ) or "- No source passed the evidence threshold"
        reply = (f"{result['answer']}\n\n**Confidence:** {result['confidence']:.1%} | "
                 f"**Chunks used:** {result['chunks_used']} | **Generator:** {ENGINE.generation_backend}\n\n"
                 f"**Sources**\n{source_text}")
        history += [{"role": "user", "content": message.strip()}, {"role": "assistant", "content": reply}]
        SESSION_LOG.append({"question": message.strip(), **result, "retrieval_backend": ENGINE.retrieval_backend,
                            "generation_backend": ENGINE.generation_backend})
        return history, "", f"Questions asked: **{len(SESSION_LOG)}**"
    except Exception as exc:
        traceback.print_exc()
        history += [{"role": "user", "content": message.strip()},
                    {"role": "assistant", "content": f"Error: {type(exc).__name__}: {exc}"}]
        return history, "", f"Questions asked: **{len(SESSION_LOG)}**"

def export_log():
    if not SESSION_LOG:
        raise gr.Error("Ask at least one question first.")
    path = Path(tempfile.gettempdir()) / "full_rag_session_log.json"
    path.write_text(json.dumps(SESSION_LOG, indent=2), encoding="utf-8")
    return str(path)

CSS = ".gradio-container{max-width:1350px!important;margin:auto!important}.hero{background:linear-gradient(100deg,#003B70,#00A6B2);color:white;padding:18px;border-radius:16px;text-align:center}.metric{border-left:5px solid #00A6B2;padding-left:12px}"
with gr.Blocks(title="Full RAG Studio", css=CSS) as demo:
    gr.Markdown("# Full RAG Studio — Amended Pfizer Capstone\nUpload multiple digital or scanned PDFs, then ask 10 or more questions manually.", elem_classes="hero")
    with gr.Row():
        with gr.Column(scale=1):
            files = gr.File(label="Upload one or more PDFs", file_types=[".pdf"], file_count="multiple")
            process_button = gr.Button("Process and Index", variant="primary")
            status = gr.Markdown("No documents indexed.", elem_classes="metric")
            document_type = gr.Dropdown(DOCUMENT_TYPES, value="All", label="Document type filter")
            auto_route = gr.Checkbox(True, label="Auto-route by question")
            k = gr.Slider(1, 8, value=4, step=1, label="Chunks to retrieve")
            threshold = gr.Slider(0, 1, value=0.05, step=0.01, label="Abstention threshold")
            counter = gr.Markdown("Questions asked: **0**")
            export_button = gr.Button("Export Q&A session")
            download = gr.File(label="Session JSON", interactive=False)
            clear = gr.Button("Clear chat")
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Grounded answers", type="messages", height=520)
            question = gr.Textbox(label="Question", placeholder="Ask one question at a time", lines=2)
            send = gr.Button("Ask", variant="primary")
    process_button.click(process_ui, files, [status, chatbot, question, counter], show_progress="full")
    send.click(chat_ui, [question, chatbot, k, document_type, auto_route, threshold], [chatbot, question, counter], show_progress="full")
    question.submit(chat_ui, [question, chatbot, k, document_type, auto_route, threshold], [chatbot, question, counter], show_progress="full")
    clear.click(lambda: ([], "", "Questions asked: **0**"), outputs=[chatbot, question, counter])
    export_button.click(export_log, outputs=download)

if __name__ == "__main__":
    demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/10/28 10:33:08 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: connect: connection refused


<IPython.core.display.Javascript object>

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cpu


## Pfizer supplier-pack test protocol
1. Upload the AKTA ready certificate plus any available supplier certificate, specification, declaration, or scanned page.
2. Process and index the documents.
3. Ask at least 10 questions manually, including one scanned-page question, one cross-document question, and one unsupported question.
4. Export the Q&A session JSON.
5. Score each answer against expected document, page, answer, and evidence before reporting Recall@k, MRR, answer accuracy, citation accuracy, latency, and the honest failure.